# What is concurrency?

# Experiment 1: Creating and observing threads

A normal Python program already has one thread: the main thread.
    
    Python Process
    │
    ├── Main Thread
    │
    │   execute statement 1
    │   execute statement 2
    │   execute statement 3
    │
    └── Shared Process Memory
        ├── objects
        ├── globals
        ├── heap
        └── imported modules

    All threads share:
        process memory
        Python objects
        globals
        heap
    
    Each thread has its own:
        call stack
        current execution position

In [1]:
import threading
import time
from datetime import datetime


def log(message):
    """
    Small helper so that every message shows:
    - current time
    - current thread name
    - what that thread is doing
    """

    current_thread = threading.current_thread()
    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(
        f"{timestamp} | "
        f"{current_thread.name:<12} | "
        f"{message}"
    )


def worker(task_name, duration):
    """
    Function that will be executed by a worker thread.
    """

    log(f"START {task_name}")

    # sleep simulates some slow operation such as:
    # - network request
    # - database query
    # - file read
    # - waiting for an external service
    time.sleep(duration)

    log(f"END   {task_name}")


# ---------------------------------------------------------
# STEP 1: Observe the main thread
# ---------------------------------------------------------

log("Program started")


# ---------------------------------------------------------
# STEP 2: Create thread objects
#
# Important:
# Creating a Thread object does NOT start the thread.
# ---------------------------------------------------------

thread1 = threading.Thread(
    target=worker,
    args=("Task-A", 3),
    name="Worker-1"
)

thread2 = threading.Thread(
    target=worker,
    args=("Task-B", 2),
    name="Worker-2"
)


log("Thread objects created")


# ---------------------------------------------------------
# STEP 3: Start the threads
#
# start() tells Python:
# "This thread is now eligible to execute."
# ---------------------------------------------------------

thread1.start()

log("Worker-1 started")

thread2.start()

log("Worker-2 started")


# ---------------------------------------------------------
# STEP 4: Main thread continues executing
#
# start() does NOT wait for the worker to finish.
# ---------------------------------------------------------

log("Main thread continues doing its own work")

time.sleep(1)

log("Main thread finished its small task")


# ---------------------------------------------------------
# STEP 5: Wait for worker threads
# join() means:
# "The current thread should wait until this other
# thread finishes."
# Here the current thread is MainThread.
# ---------------------------------------------------------

log("Main thread waiting for Worker-1")

thread1.join()

log("Worker-1 has finished")


log("Main thread waiting for Worker-2")

thread2.join()

log("Worker-2 has finished")


log("Program finished")

06:11:07.779 | MainThread   | Program started
06:11:07.784 | MainThread   | Thread objects created
06:11:07.785 | Worker-1     | START Task-A
06:11:07.786 | MainThread   | Worker-1 started
06:11:07.786 | Worker-2     | START Task-B
06:11:07.786 | MainThread   | Worker-2 started
06:11:07.786 | MainThread   | Main thread continues doing its own work
06:11:08.790 | MainThread   | Main thread finished its small task
06:11:08.790 | MainThread   | Main thread waiting for Worker-1
06:11:09.791 | Worker-2     | END   Task-B
06:11:10.792 | Worker-1     | END   Task-A
06:11:10.793 | MainThread   | Worker-1 has finished
06:11:10.793 | MainThread   | Main thread waiting for Worker-2
06:11:10.794 | MainThread   | Worker-2 has finished
06:11:10.794 | MainThread   | Program finished


In [3]:
str(datetime.now())

'2026-09-02 06:17:21.946877'

In [34]:
def log(message):
    curr_time = str(datetime.now())
    current_thread = threading.current_thread()
    active_threads = threading.active_count()
    print(f"{curr_time} | {message} | thread = {current_thread.name} | active threads: {active_threads} ")


def some_task(task_name, duration):
    log(f"Starting task: {task_name}")
    time.sleep(duration)

    log(f"finished task: {task_name}")

In [29]:
## sequential operation

log("Starting the breakfast process")

some_task("making coffee", 5)
some_task("making toast", 10)


log("finished the breakfast process")


2026-09-02 06:40:44.802125 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 06:40:44.802269 | Starting task: making coffee | thread = MainThread | active threads: 9 
2026-09-02 06:40:49.806632 | finished task: making coffee | thread = MainThread | active threads: 9 
2026-09-02 06:40:49.807320 | Starting task: making toast | thread = MainThread | active threads: 9 
2026-09-02 06:40:59.808496 | finished task: making toast | thread = MainThread | active threads: 9 
2026-09-02 06:40:59.809165 | finished the breakfast process | thread = MainThread | active threads: 9 


In [35]:
## threaded operation

from threading import Thread


log("Starting the breakfast process")

## create a thread

log("Creating the thread objects")

coffee_thread = threading.Thread(target = some_task, args =  ("making coffee", 5), name = "coffee_thread") 
toast_thread = threading.Thread(target = some_task, args =  ("making toast", 10), name = "toast_thread") 

log("Created the thread objects")


log("Starting the thread objects!!")

coffee_thread.start()
log("coffee thread started!!")

toast_thread.start()
log("toast thread started!!")


## try adding thread.join

coffee_thread.join()
toast_thread.join()

log("finished the breakfast process!!!")

2026-09-02 07:04:08.438949 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439077 | Creating the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439295 | Created the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439362 | Starting the thread objects!! | thread = MainThread | active threads: 9 
2026-09-02 07:04:08.439588 | Starting task: making coffee | thread = coffee_thread | active threads: 10 
2026-09-02 07:04:08.439846 | coffee thread started!! | thread = MainThread | active threads: 10 
2026-09-02 07:04:08.439987 | Starting task: making toast | thread = toast_thread | active threads: 11 
2026-09-02 07:04:08.440137 | toast thread started!! | thread = MainThread | active threads: 11 
2026-09-02 07:04:13.444007 | finished task: making coffee | thread = coffee_thread | active threads: 11 
2026-09-02 07:04:18.443889 | finished task: making toast | thread = toast_thread | active 

In [36]:
print(dir(threading))

['Barrier', 'BoundedSemaphore', 'BrokenBarrierError', 'Condition', 'Event', 'ExceptHookArgs', 'Lock', 'RLock', 'Semaphore', 'TIMEOUT_MAX', 'Thread', 'ThreadError', 'Timer', 'WeakSet', '_CRLock', '_DeleteDummyThreadOnDel', '_DummyThread', '_HAVE_THREAD_NATIVE_ID', '_LockType', '_MainThread', '_PyRLock', '_RLock', '_SHUTTING_DOWN', '_ThreadHandle', '__all__', '__builtins__', '__cached__', '__doc__', '__excepthook__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_active', '_active_limbo_lock', '_after_fork', '_allocate_lock', '_count', '_counter', '_daemon_threads_allowed', '_dangling', '_deque', '_enumerate', '_get_main_thread_ident', '_is_main_interpreter', '_limbo', '_main_thread', '_make_invoke_excepthook', '_make_thread_handle', '_newname', '_os', '_profile_hook', '_register_atexit', '_shutdown', '_start_joinable_thread', '_sys', '_thread_local_info', '_thread_shutdown', '_threading_atexits', '_time', '_trace_hook', 'activeCount', 'active_count', 'currentThread',

In [17]:
threading.active_count()

9

#### see the threads a little better

In [31]:
import threading
import time
import os
from datetime import datetime


def show_threads(label):
    """
    Display what Python currently knows about its threads.
    """

    print(f"\n{'=' * 70}")
    print(label)
    print(f"{'=' * 70}")

    print(f"Process ID (PID): {os.getpid()}")
    print(f"Active thread count: {threading.active_count()}")

    for thread in threading.enumerate():
        print(
            f"  Thread name={thread.name!r}, "
            f"ident={thread.ident}, "
            f"native_id={thread.native_id}, "
            f"alive={thread.is_alive()}"
        )


def some_task():
    print(
        f"\nWorker executing: "
        f"name={threading.current_thread().name}, "
        f"ident={threading.current_thread().ident}, "
        f"native_id={threading.current_thread().native_id}"
    )

    time.sleep(50)

# ---------------------------------------------------------
# STEP 1 - Inspect the process before creating our Thread object.
# ---------------------------------------------------------

show_threads("1. BEFORE creating Thread object")

# ---------------------------------------------------------
# STEP 2 - Create a Thread object.
# This does NOT start an OS thread yet.
# ---------------------------------------------------------

worker = threading.Thread(
    target=some_task,
    name="CoffeeThread"
)

show_threads("2. AFTER creating Thread object")
print(
    "\nPython Thread object says:"
    f" name={worker.name!r},"
    f" ident={worker.ident},"
    f" native_id={worker.native_id},"
    f" alive={worker.is_alive()}"
)


# ---------------------------------------------------------
# STEP 3 - Actually start the thread.
# ---------------------------------------------------------

print("\nCalling worker.start()...")

worker.start()


show_threads("3. AFTER worker.start()")


# ---------------------------------------------------------
# STEP 4 - Wait for the worker to finish.
# ---------------------------------------------------------

worker.join()


show_threads("4. AFTER worker.join()")


1. BEFORE creating Thread object
Process ID (PID): 50970
Active thread count: 9
  Thread name='MainThread', ident=8269492800, native_id=580585, alive=True
  Thread name='IOPub', ident=6140538880, native_id=580693, alive=True
  Thread name='Heartbeat', ident=6157365248, native_id=580694, alive=True
  Thread name='Thread-2 (_watch_pipe_fd)', ident=6175338496, native_id=580697, alive=True
  Thread name='Thread-3 (_watch_pipe_fd)', ident=6192164864, native_id=580698, alive=True
  Thread name='Control', ident=6208991232, native_id=580699, alive=True
  Thread name='Shell channel', ident=6225817600, native_id=580700, alive=True
  Thread name='IPythonHistorySavingThread', ident=6242643968, native_id=580721, alive=True
  Thread name='Thread-1', ident=6260043776, native_id=580725, alive=True

2. AFTER creating Thread object
Process ID (PID): 50970
Active thread count: 9
  Thread name='MainThread', ident=8269492800, native_id=580585, alive=True
  Thread name='IOPub', ident=6140538880, native_id=

In [ ]:
##### ident versus native_id


     - On macOS/Linux/Windows, native_id corresponds to the thread identifier assigned by the underlying operating system.
     - start() is the point at which the thread actually comes into existence as an executing thread.

#### seeing the threads in terminal 
get the pid - `os.getpid()`

ps -M <PID>

#### observing the thread lifecycle

In [32]:
import threading
import time
import os
from datetime import datetime


def log(message):
    """
    Print useful information about the current execution context.
    """

    now = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    current = threading.current_thread()
    print(
        f"{now} | "
        f"PID={os.getpid()} | "
        f"Thread={current.name} | "
        f"native_id={current.native_id} | "
        f"{message}"
    )


def worker():
    """
    Worker function executed by our thread.
    """

    log("Worker function STARTED")

    # Keep the thread alive for 10 seconds.
    # During this period the thread is alive, but sleeping.
    time.sleep(10)

    log("Worker function FINISHED")


# ==========================================================
# STEP 1: Create the Thread object
# ==========================================================

worker_thread = threading.Thread(
    target=worker,
    name="Worker-1"
)

print("\n--- AFTER CREATING THREAD OBJECT ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


# ==========================================================
# STEP 2: Start the thread
# ==========================================================

print("\n--- CALLING start() ---")

worker_thread.start()

print("\n--- IMMEDIATELY AFTER start() ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


# ==========================================================
# STEP 3: MainThread continues
# ==========================================================

print("\n--- MAIN THREAD CONTINUES ---")

for i in range(3):

    print(
        f"MainThread doing work {i + 1} | "
        f"Worker alive={worker_thread.is_alive()}"
    )

    time.sleep(1)


# ==========================================================
# STEP 4: Wait for the worker
# ==========================================================

print("\n--- MAIN THREAD CALLING join() ---")

worker_thread.join()

print("\n--- AFTER join() ---")

print("name      :", worker_thread.name)
print("ident     :", worker_thread.ident)
print("native_id :", worker_thread.native_id)
print("alive     :", worker_thread.is_alive())


--- AFTER CREATING THREAD OBJECT ---
name      : Worker-1
ident     : None
native_id : None
alive     : False

--- CALLING start() ---
07:01:27.156 | PID=50970 | Thread=Worker-1 | native_id=769079 | Worker function STARTED

--- IMMEDIATELY AFTER start() ---
name      : Worker-1
ident     : 6344175616
native_id : 769079
alive     : True

--- MAIN THREAD CONTINUES ---
MainThread doing work 1 | Worker alive=True
MainThread doing work 2 | Worker alive=True
MainThread doing work 3 | Worker alive=True

--- MAIN THREAD CALLING join() ---
07:01:37.160 | PID=50970 | Thread=Worker-1 | native_id=769079 | Worker function FINISHED

--- AFTER join() ---
name      : Worker-1
ident     : 6344175616
native_id : 769079
alive     : False


In [40]:
## threaded operation

from threading import Thread


log("Starting the breakfast process")

## create a thread

log("Creating the thread objects")

coffee_thread = threading.Thread(target = some_task, args =  ("making coffee", 5), name = "coffee_thread") 
# toast_thread = threading.Thread(target = some_task, args =  ("making toast", 10), name = "toast_thread") 

log("Created the thread objects")


log("Starting the thread objects!!")

coffee_thread.start()
log("coffee thread started!!")

# toast_thread.start()
# log("toast thread started!!")


## try adding thread.join

for i in range(10): # this is for checking the status of coffee thread whether its active or not
    log(f"***status of coffee thread: , {coffee_thread.is_alive() = }")
    time.sleep(1)


log("finished the breakfast process")




2026-09-02 07:10:19.066784 | Starting the breakfast process | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.067364 | Creating the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.067841 | Created the thread objects | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.068031 | Starting the thread objects!! | thread = MainThread | active threads: 9 
2026-09-02 07:10:19.068312 | Starting task: making coffee | thread = coffee_thread | active threads: 10 
2026-09-02 07:10:19.069915 | coffee thread started!! | thread = MainThread | active threads: 10 
2026-09-02 07:10:19.070336 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active threads: 10 
2026-09-02 07:10:20.072551 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active threads: 10 
2026-09-02 07:10:21.074183 | ***status of coffee thread: , coffee_thread.is_alive() = True | thread = MainThread | active th

#### what about GIL ?

In [41]:
import sys

print("Python version:", sys.version)

# Available in Python 3.13+.
# True  -> this interpreter has the GIL enabled.
# False -> this is a free-threaded build.
if hasattr(sys, "_is_gil_enabled"):
    print("GIL enabled:", sys._is_gil_enabled())
else:
    print("This Python version does not expose _is_gil_enabled().")

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
GIL enabled: True


#### How are the threads scheduled? 

In [42]:
import threading
import time


def cpu_task(name, seconds):
    """
    Perform CPU-intensive Python work for approximately
    'seconds' seconds.

    There is deliberately no time.sleep() here.
    """

    thread = threading.current_thread()

    print(
        f"{name} STARTED | "
        f"thread={thread.name} | "
        f"native_id={thread.native_id}"
    )
    
    end_time = time.perf_counter() + seconds
    counter = 0
    # Keep doing Python work until the requested duration passes.
    while time.perf_counter() < end_time:
        counter += 1
    print(
        f"{name} FINISHED | "
        f"thread={thread.name} | "
        f"counter={counter:,}"
    )


# ---------------------------------------------------------
# Create two CPU-bound worker threads.
# ---------------------------------------------------------

thread1 = threading.Thread(
    target=cpu_task,
    args=("Task-A", 5),
    name="Worker-A"
)

thread2 = threading.Thread(
    target=cpu_task,
    args=("Task-B", 5),
    name="Worker-B"
)


# ---------------------------------------------------------
# Start both threads.
# ---------------------------------------------------------

start = time.perf_counter()
thread1.start()
thread2.start()


# ---------------------------------------------------------
# Wait for both workers.
# ---------------------------------------------------------

thread1.join()
thread2.join()

elapsed = time.perf_counter() - start

print(f"\nTotal elapsed time: {elapsed:.2f} seconds")

Task-A STARTED | thread=Worker-A | native_id=836116
Task-B STARTED | thread=Worker-B | native_id=836119
Task-A FINISHED | thread=Worker-A | counter=36,011,393
Task-B FINISHED | thread=Worker-B | counter=35,693,464

Total elapsed time: 5.02 seconds


# Experiment 2: Compare sequential and threaded execution

In [ ]:
import threading
import time


def slow_task(name, duration):
    print(f"{name} started")

    time.sleep(duration)

    print(f"{name} finished")


# =========================================================
# SEQUENTIAL VERSION
# =========================================================

print("\nSEQUENTIAL EXECUTION")

start = time.perf_counter()

slow_task("Task-1", 2)
slow_task("Task-2", 2)
slow_task("Task-3", 2)

end = time.perf_counter()

print(f"Sequential time: {end - start:.2f} seconds")


# =========================================================
# MULTITHREADED VERSION
# =========================================================

print("\nMULTITHREADED EXECUTION")

start = time.perf_counter()

threads = []

for i in range(3):

    thread = threading.Thread(
        target=slow_task,
        args=(f"Task-{i + 1}", 2)
    )

    threads.append(thread)

    thread.start()


# Wait until every worker completes
for thread in threads:
    thread.join()


end = time.perf_counter()

print(f"Threaded time: {end - start:.2f} seconds")

# Experiment 3 : Race Conditions


The central problem is very simple:

`Multiple threads can access the same piece of shared state, and the final result can depend on the timing and interleaving of their execution.`

That situation is called a race condition.

In [44]:
counter = 0

counter += 1
counter += 1
counter += 1

print(counter)

3


In [47]:
import threading
import time


counter = 0


def increment_counter():
    global counter
    # Read the shared value.
    current_value = counter
    # Deliberately create a window where another thread can execute before we write the result back.
    time.sleep(0.001)
    # Write the updated value.
    counter = current_value + 1


threads = []

# Create 100 threads.
for _ in range(100):

    thread = threading.Thread(
        target=increment_counter
    )

    threads.append(thread)
    thread.start()



# Wait for every thread to finish.
for thread in threads:
    thread.join()


print("Expected:", 100)
print("Actual:  ", counter)

Expected: 100
Actual:   9


    Thread A       Thread B       Thread C
    
    read 0
                   read 0
                                  read 0
    
    write 1
                   write 1
                                  write 1


- three threads worked here but we ended up with counter = 1 instead of counter = 3
- Why is this called a "race"? - Because there is a competition between threads over the timing of operations.

    The correctness of the program depends on the relative timing/interleaving of concurrent operations on shared state.

    
    This is perfectly fine:
    
        Thread A → read database
        Thread B → read database
    They aren't necessarily racing.
    
    But this can race:
    
        Thread A → read inventory = 1
        Thread B → read inventory = 1
        Thread A → decrement inventory
        Thread B → decrement inventory
    
    because both operations depend on shared mutable state.

#### critical section.

A critical section is a portion of code that accesses shared state and must be executed with the required synchronization so that multiple threads don't interfere with one another.

In our example, this is the dangerous sequence:

```
read counter
add 1
write counter
```

We want that entire operation to behave as one logical unit.

Conceptually:

                  Critical Section

              ┌──────────────────────┐
              │ read counter         │
              │ add 1                │
              │ write counter        │
              └──────────────────────┘
We dont want

```
Thread A enters
Thread A reads

Thread B enters
Thread B reads

Thread A writes
Thread B writes
```


We want

```
Thread A
   |
   v
[ read → increment → write ]
   |
   v
Thread B
   |
   v
[ read → increment → write ]
```

#### What is a lock?

A lock is a synchronization mechanism that allows us to say:

"Only one thread at a time may enter this protected section."

In [48]:
import threading
import time


counter = 0

# Create one lock protecting the shared counter.
counter_lock = threading.Lock()


def increment_counter():
    global counter

    # Acquire the lock before touching the shared state.
    with counter_lock:
        current_value= counter
        # We deliberately keep the sleep here to demonstrate that even if this thread pauses, another thread cannot enter this critical section.
        time.sleep(0.001)
        counter = current_value + 1


threads = []
for _ in range(100):
    thread = threading.Thread(target=increment_counter)
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

print("Expected:", 100)
print("Actual:  ", counter)

Expected: 100
Actual:   100


 - The lock therefore protects the atomicity of the logical operation.
 - What is atomicity? --> An operation is atomic if it appears to happen as one indivisible operation from the perspective of other threads. In other words, another thread cannot observe the operation halfway through or interleave its own conflicting operation in the middle.

Consider this example:

```counter += 1```

At the conceptual level, this involves multiple steps:
```
read counter
add 1
write counter
```

Suppose ```counter == 10```.

Without synchronization, two threads could interleave their operations like this:

    Thread A                 Thread B
    
    read 10
                             read 10
    
    calculate 11
                             calculate 11
    
    write 11
                             write 11

The expected result after two increments is 12, but we end up with 11.

The problem is not that the individual read or write necessarily failed. The problem is that the whole read-modify-write operation was not atomic.

We therefore want this:

    Thread A
    ┌─────────────────────┐
    │ read → modify → write│
    └─────────────────────┘
    
    Thread B
                           ┌─────────────────────┐
                           │ read → modify → write│
                           └─────────────────────┘

rather than allowing the operations to interleave.

    A useful way to think about atomicity is: Atomicity is about whether an operation can be observed or interfered with halfway through. A lock is one mechanism for providing that guarantee.
    
    There is another important distinction: atomicity and thread safety are not synonyms. Atomicity is a property of an individual operation or group of operations. Thread safety is the broader property that a component behaves correctly when accessed concurrently. We will see later that achieving thread safety may require more than simply putting a lock around one statement.

### application of threading.lock

In [50]:
class Inventory:
    def __init__(self):
        self.stock = 1

    def purchase(self):
        if self.stock > 0:
            self.stock -= 1
            return True

        return False

    Customer A                     Customer B
    
    check stock > 0
                                   check stock > 0
    
    both see stock = 1
    
    decrement stock
                                   decrement stock

You can potentially sell one item twice. This is the same conceptual problem as our counter.
The critical section is:

        check whether stock exists
                +
        decrement stock

Those operations must be treated as one synchronized operation.

Thread safe implementation

```python
with self.lock:
    if self.stock > 0:
        self.stock -= 1
        return True

    return False

#### some more examples


Identify the shared mutable state and determine which sequence of operations must be performed atomically.

##### Example 1: Shared counter

In [ ]:
## Without synchronization:
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1


        import threading

## with lock
class Counter:
    def __init__(self):
        self.value = 0
        self.lock = threading.Lock()

    def increment(self):
        with self.lock:
            self.value += 1

##### Example 2: Bank account

In [ ]:
class BankAccount:
    def __init__(self):
        self.balance = 100

    def withdraw(self, amount):
        if self.balance >= amount:
            self.balance -= amount
            return True

        return False





    Imagine two threads simultaneously withdraw $80.
    
    They could see:
    
    Initial balance = $100
    
    Thread A: check $100 >= $80 → True
    Thread B: check $100 >= $80 → True
    
    Thread A: balance = $20
    Thread B: balance = -$60

In [56]:
## The synchronized version is:
import threading


class BankAccount:
    def __init__(self):
        self.balance = 100
        self.lock = threading.Lock()

    def withdraw(self, amount):
        with self.lock:
            if self.balance >= amount:
                self.balance -= amount
                return True

            return False

##### Example 3: Inventory

In [ ]:
class Inventory:
    def __init__(self):
        self.stock = 1

    def purchase(self):
        if self.stock > 0:
            self.stock -= 1
            return True

        return False

In [ ]:
import threading


class Inventory:
    def __init__(self):
        self.stock = 1
        self.lock = threading.Lock()

    def purchase(self):
        with self.lock:
            if self.stock > 0:
                self.stock -= 1
                return True

            return False

##### Example 4: Booking a seat

In [ ]:
class Seat:
    def __init__(self):
        self.available = True

    def book(self):
        if self.available:
            self.available = False
            return True

        return False

without syncronization

    Thread A                  Thread B
    
    available?
    → True                    available?
                              → True
    
    book seat                 book seat

In [58]:
# synchronized version
import threading


class Seat:
    def __init__(self):
        self.available = True
        self.lock = threading.Lock()

    def book(self):
        with self.lock:
            if self.available:
                self.available = False
                return True

            return False

##### Example 5: Updating related state

In [ ]:
class ShoppingCart:
    def __init__(self):
        self.total_items = 0
        self.total_value = 0

    def add_item(self, price):
        self.total_items += 1
        self.total_value += price

A reader could potentially observe:

    total_items = 5
    total_value = 400

while another thread is halfway through an update and the state temporarily represents an inconsistent combination.

In [ ]:
import threading


class ShoppingCart:
    def __init__(self):
        self.total_items = 0
        self.total_value = 0
        self.lock = threading.Lock()

    def add_item(self, price):
        with self.lock:
            self.total_items += 1
            self.total_value += price

The important concept here is consistency of shared state. Sometimes the critical section exists not because one individual variable is problematic, but because several variables collectively represent one logical state.

# Experiment 4: acquire() and release()

We have so far used:

```python
with lock:
    # critical section
```
    
Python's with syntax is convenient, but underneath it is essentially managing:

```python
lock.acquire()

try:
    # critical section
finally:
    lock.release()
```

Let's understand these operations directly.

```acquire()``` means:

    Attempt to acquire ownership of the lock.
    If nobody currently owns it, the calling thread obtains it immediately.
    If another thread already owns it, the calling thread waits until the lock becomes available.

```release()``` means:

    Release the lock so that another waiting thread can acquire it.


Basic Pattern:
```python
lock.acquire()

try:
    # Critical section
finally:
    lock.release()
```

For example:

```python
import threading


lock = threading.Lock()
lock.acquire()

try:
    print("Thread entered critical section")

    # Shared state modification happens here.

finally:
    lock.release()

print("Thread left critical section")
```

    Thread A
       |
       | acquire()
       v
    ┌───────────────────┐
    │   LOCKED          │
    │                   │
    │ critical section  │
    │                   │
    └─────────┬─────────┘
              |
              | release()
              v
           UNLOCKED

What happens when thread A and thread B both try to access critical section    
    
    Thread A                    Thread B
        
        acquire()
            |
            v
         LOCKED
            |
         critical section       acquire()
                                    |
                                    v
                                 WAITING
                                    |
                                    |
         release()                  |
            |                       |
            └──────────────────────>|
                                    |
                               acquires lock
                                    |
                                    v
                             critical section

The important difference is that with is the safer and more convenient way to express the pattern because Python guarantees that the lock is released when the block is exited, including when an exception occurs.

These are conceptually equivalent:

```python
lock.acquire()

try:
    # Critical section
    update_shared_state()

finally:
    lock.release()
```

and:

```python
with lock:
    # Critical section
    update_shared_state()
```

The second form is essentially using the lock as a context manager. Under the hood, Python calls acquire() when entering the with block and release() when leaving it.

So why would we ever use acquire() directly?

Because acquire() gives you more control over the locking behavior.

For example, you can ask Python not to wait if the lock is currently unavailable:

```python
acquired = lock.acquire(blocking=False)

if acquired:
    try:
        # We successfully acquired the lock.
        update_shared_state()
    finally:
        lock.release()
else:
    # Someone else currently owns the lock.
    print("Could not acquire lock; doing something else.")
```

Here, ```with lock```: cannot directly express the same "try once and don't wait" behavior.

You can also specify a timeout:

```python
acquired = lock.acquire(timeout=2)

if acquired:
    try:
        update_shared_state()
    finally:
        lock.release()
else:
    print("Lock was not acquired within 2 seconds.")
```

So the practical rule is:

```
with lock:
    ...
```
    
is the normal choice when you simply want to protect a critical section.
```
lock.acquire(...)
lock.release()
```

is useful when you need explicit control over acquisition, such as non-blocking acquisition, timeouts, or more complicated synchronization logic.

One subtle but important point: never casually write this:

```python
lock.acquire()

update_shared_state()

lock.release()
```
    
because if update_shared_state() raises an exception, release() may never execute, leaving the lock permanently acquired. That can cause other threads to wait forever. If you need manual acquire()/release(), use try/finally.

# Experiment 5: Lock contention -  what happens when two threads call acquire()

In [62]:
# Two threads competing for one lock

import threading
import time
from datetime import datetime


lock = threading.Lock()


def log(message):
    """
    Print the current time and thread name so that we can
    observe the ordering of events.
    """

    timestamp = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    thread = threading.current_thread()

    print(
        f"{timestamp} | "
        f"{thread.name:<12} | "
        f"{message}"
    )


def worker():
    log("About to call lock.acquire()")

    # This thread will wait here if another thread already
    # owns the lock.
    lock.acquire()

    try:
        log("Successfully acquired the lock")

        # Hold the lock for 5 seconds.
        log("Entering critical section")
        time.sleep(5)
        log("Leaving critical section")

    finally:
        # Always release the lock.
        lock.release()

        log("Released the lock")


thread_a = threading.Thread(target=worker,name="Worker-A")
thread_b = threading.Thread(target=worker,name="Worker-B")


log("Starting Worker-A")
thread_a.start()

# Give Worker-A enough time to acquire the lock before starting Worker-B.
time.sleep(0.5)

log("Starting Worker-B")
thread_b.start()

thread_a.join()
thread_b.join()

log("Both workers finished")

08:50:35.059 | MainThread   | Starting Worker-A
08:50:35.060 | Worker-A     | About to call lock.acquire()
08:50:35.060 | Worker-A     | Successfully acquired the lock
08:50:35.060 | Worker-A     | Entering critical section
08:50:35.562 | MainThread   | Starting Worker-B
08:50:35.563 | Worker-B     | About to call lock.acquire()
08:50:40.064 | Worker-A     | Leaving critical section
08:50:40.066 | Worker-A     | Released the lock
08:50:40.066 | Worker-B     | Successfully acquired the lock
08:50:40.066 | Worker-B     | Entering critical section
08:50:45.072 | Worker-B     | Leaving critical section
08:50:45.074 | Worker-B     | Released the lock
08:50:45.075 | MainThread   | Both workers finished


what happened in the above code:


    Worker-A:
        acquired lock
        entered critical section
        ...
        ...
        released lock
    
    Worker-B:
        called acquire()
        WAITED
        WAITED
        WAITED
        lock became available
        acquired lock
        entered critical section


This is **lock contention**. Multiple threads want the same resource, but the resource can only be accessed by one thread at a time.

```acquire()``` can also be non-blocking

In [65]:
import threading
import time


lock = threading.Lock()


def worker_a():
    with lock:
        print("Worker-A acquired the lock")
        time.sleep(5)
        print("Worker-A releasing the lock")


def worker_b():
    time.sleep(1)

    print("Worker-B trying to acquire the lock")

    acquired = lock.acquire(blocking=False)

    if acquired:
        try:
            print("Worker-B acquired the lock")
        finally:
            lock.release()
    else:
        print("Worker-B could not acquire the lock")
        print("Worker-B will do something else")


thread_a = threading.Thread(
    target=worker_a,
    name="Worker-A"
)

thread_b = threading.Thread(
    target=worker_b,
    name="Worker-B"
)

thread_a.start()
thread_b.start()

thread_a.join()
thread_b.join()

Worker-A acquired the lock
Worker-B trying to acquire the lock
Worker-B could not acquire the lock
Worker-B will do something else
Worker-A releasing the lock


Here Worker-B doesn't wait for Worker-A.

Instead:

    Worker-A:
        acquire
        |
        | holds lock for 5 sec
        |
        release
    
    
    Worker-B:
        acquire(blocking=False)
              |
              v
          unavailable
              |
              v
          immediately continue

This is useful when waiting is undesirable and the application has an alternative course of action.

using timeout in acquire ---> ```acquire(timeout=...)```

```acquire(timeout=2)``` means try to acquire the lock, but wait at most two seconds.

In [ ]:
import threading
import time


lock = threading.Lock()


def worker_a():
    with lock:
        print("Worker-A acquired the lock")

        time.sleep(5)

        print("Worker-A releasing the lock")


def worker_b():
    time.sleep(1)

    print("Worker-B trying to acquire the lock")

    acquired = lock.acquire(timeout=2)

    if acquired:
        try:
            print("Worker-B acquired the lock")
        finally:
            lock.release()
    else:
        print("Worker-B timed out waiting for the lock")


thread_a = threading.Thread(target=worker_a)
thread_b = threading.Thread(target=worker_b)

thread_a.start()
thread_b.start()

thread_a.join()
thread_b.join()

# Lesson 6: Check-Then-Act

The check-then-act pattern is one of the most important concurrency patterns to understand because many real-world operations naturally have two steps: first we check whether some condition is true, and then we perform an action based on that observation. The problem is that, when multiple threads are involved, another thread can change the shared state between the check and the action.